# CatSkin AI - Training MobileNetV2
Jalankan cell satu per satu dari atas ke bawah.

> Pastikan **Runtime → Change runtime type → T4 GPU** sebelum mulai

# 1. Mount Google Drive
Menghubungkan Google Colab ke Google Drive agar bisa mengakses dataset dan menyimpan model yang sudah ditraining.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')  # Akan muncul popup — klik 'Connect to Google Drive' lalu pilih akun

# 2. Path Dataset
Menentukan lokasi folder dataset di Google Drive dan folder output untuk menyimpan model hasil training. Pastikan struktur folder sudah benar: `Cat_Skin_Diagnosa/train/`, `Cat_Skin_Diagnosa/val/`, `Cat_Skin_Diagnosa/test/` — masing-masing berisi subfolder per kelas.

In [ ]:
import os

DATASET_DIR  = '/content/drive/MyDrive/Cat_Skin_Diagnosa'  # <-- GANTI jika nama folder berbeda
MODEL_OUTPUT = '/content/drive/MyDrive/cnn_model.h5'       # Path untuk menyimpan model terbaik

# Cek isi folder — pastikan nama subfolder kelas muncul dan urutannya benar
print('Train classes:', sorted(os.listdir(os.path.join(DATASET_DIR, 'train'))))
print('Val   classes:', sorted(os.listdir(os.path.join(DATASET_DIR, 'val'))))
print('Test  classes:', sorted(os.listdir(os.path.join(DATASET_DIR, 'test'))))

# 3. Import Library
Mengimpor semua library yang dibutuhkan. TensorFlow sebagai framework utama, MobileNetV2 sebagai arsitektur CNN, dan beberapa callback untuk mengontrol proses training secara otomatis.

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2                          # Arsitektur CNN pre-trained ImageNet
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input        # Normalisasi khusus MobileNetV2: pixel → range -1 s/d 1
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout     # Layer untuk custom classifier head
from tensorflow.keras.models import Model                                       # Untuk merakit base model + custom head
from tensorflow.keras.optimizers import Adam                                    # Optimizer adaptif, cocok untuk transfer learning
from tensorflow.keras.callbacks import (
    ModelCheckpoint,   # Simpan otomatis model terbaik selama training
    EarlyStopping,     # Hentikan training jika tidak ada peningkatan
    ReduceLROnPlateau  # Kurangi learning rate otomatis jika stagnan
)

print('TensorFlow:', tf.__version__)
print('GPU       :', tf.config.list_physical_devices('GPU'))  # Harus ada GPU — jika kosong, ganti runtime ke T4

# 4. Konfigurasi
Mendefinisikan semua hyperparameter training dalam satu tempat agar mudah diubah. `CLASS_NAMES` wajib disamakan dengan `CLASS_LABELS` di `backend/app/cnn_service.py` — jika urutannya berbeda, prediksi akan salah label meski akurasi tinggi.

In [ ]:
IMG_SIZE    = (224, 224)  # Ukuran input wajib MobileNetV2 — jangan diubah
BATCH_SIZE  = 32          # Jumlah gambar per batch — turunkan ke 16 jika RAM Colab habis
EPOCHS      = 50          # Maksimum epoch fase 2 — EarlyStopping akan berhenti lebih awal
LR          = 1e-4        # Learning rate awal untuk fase 1 (training head)
NUM_CLASSES = 5           # Jumlah kelas penyakit kulit

# Urutan kelas HARUS konsisten dengan folder dataset dan CLASS_LABELS di cnn_service.py
CLASS_NAMES = ['Dermatitis', 'Flea Allergy', 'Ringworm', 'Scabies', 'Healthy']

# 5. Load Dataset
Memuat gambar dari folder `train/`, `val/`, dan `test/` menggunakan `tf.data` pipeline. Pipeline ini lebih cepat dari `ImageDataGenerator` karena load data secara paralel dan menyiapkan batch berikutnya di background saat GPU sedang training.

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATASET_DIR, 'train'),  # Folder gambar training
    image_size=IMG_SIZE,                  # Resize semua gambar ke 224x224
    batch_size=BATCH_SIZE,
    label_mode='categorical',             # Label sebagai one-hot vector (misal: [1,0,0,0,0])
    class_names=CLASS_NAMES,              # Paksa urutan kelas sesuai CLASS_NAMES di atas
    shuffle=True,
    seed=42                               # Seed agar hasil shuffle reproducible
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATASET_DIR, 'val'),     # Folder gambar validasi
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    class_names=CLASS_NAMES,
    shuffle=False                         # Validasi tidak perlu di-shuffle
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATASET_DIR, 'test'),    # Folder gambar test — hanya dipakai di evaluasi final
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    class_names=CLASS_NAMES,
    shuffle=False
)

print('Urutan kelas:', train_ds.class_names)  # Pastikan urutan sesuai CLASS_NAMES
print('Train batches:', len(train_ds))
print('Val   batches:', len(val_ds))
print('Test  batches:', len(test_ds))

# 6. Augmentasi & Preprocessing
Augmentasi membuat model lebih robust dengan menciptakan variasi gambar secara acak saat training. `preprocess_input` MobileNetV2 mengubah nilai pixel dari range 0–255 menjadi -1 s/d 1 — ini wajib karena MobileNetV2 dilatih dengan range tersebut. Augmentasi hanya diterapkan ke training, bukan validasi dan test.

In [ ]:
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),  # Cermin gambar secara horizontal
    tf.keras.layers.RandomRotation(0.15),      # Rotasi acak ±15%
    tf.keras.layers.RandomZoom(0.1),           # Zoom in/out acak ±10%
    tf.keras.layers.RandomBrightness(0.1),     # Variasi kecerahan ±10%
    tf.keras.layers.RandomContrast(0.1),       # Variasi kontras ±10%
])

# Training: augmentasi dulu, lalu preprocess_input
train_ds = train_ds.map(lambda x, y: (augmentation(x, training=True), y),
                        num_parallel_calls=tf.data.AUTOTUNE)  # Proses paralel di background
train_ds = train_ds.map(lambda x, y: (preprocess_input(x), y),
                        num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

# Validasi & test: hanya preprocess_input, tanpa augmentasi
val_ds  = val_ds.map(lambda x, y: (preprocess_input(x), y),
                     num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.map(lambda x, y: (preprocess_input(x), y),
                      num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

# 7. Build Model
Membangun model dengan teknik Transfer Learning: menggunakan MobileNetV2 yang sudah dilatih dengan 1.4 juta gambar ImageNet sebagai feature extractor, lalu menambahkan custom classifier head di atasnya. Pada fase ini semua layer MobileNetV2 di-freeze — hanya head yang dilatih.

In [ ]:
base = MobileNetV2(input_shape=(*IMG_SIZE, 3), include_top=False, weights='imagenet')
# include_top=False : buang classifier asli MobileNetV2 (1000 kelas ImageNet)
# weights='imagenet': gunakan bobot pre-trained, bukan random
base.trainable = False  # Freeze semua layer base — hanya head yang dilatih di fase 1

x = base.output
x = GlobalAveragePooling2D()(x)      # Ubah feature map (7,7,1280) menjadi vektor (1280,)
x = Dropout(0.3)(x)                  # Dropout 30% untuk mencegah overfitting
x = Dense(128, activation='relu')(x) # Fully connected layer untuk belajar fitur spesifik kucing
x = Dropout(0.2)(x)                  # Dropout 20% tambahan
output = Dense(NUM_CLASSES, activation='softmax')(x)  # Output: probabilitas 5 kelas

model = Model(inputs=base.input, outputs=output)
model.compile(
    optimizer=Adam(LR),                  # Adam: optimizer adaptif, cocok untuk fine-tuning
    loss='categorical_crossentropy',     # Loss function standar untuk multi-class classification
    metrics=['accuracy']
)
model.summary()  # Perhatikan 'Trainable params' — seharusnya hanya head layer

# 8. Fase 1 — Training Head Layer
Melatih hanya custom head selama 10 epoch dengan base model di-freeze. Tujuannya agar head layer sudah 'hangat' sebelum fine-tuning — jika langsung fine-tune dari awal, gradien besar dari head yang random bisa merusak bobot pre-trained.

In [ ]:
callbacks = [
    # Simpan model ke Drive setiap kali val_accuracy meningkat
    ModelCheckpoint(MODEL_OUTPUT, save_best_only=True, monitor='val_accuracy', verbose=1),
    # Hentikan training jika val_accuracy tidak naik selama 7 epoch berturut-turut
    EarlyStopping(patience=7, restore_best_weights=True, monitor='val_accuracy'),
    # Kurangi LR sebesar 50% jika val_loss tidak turun selama 3 epoch (LR baru = LR * 0.5)
    ReduceLROnPlateau(factor=0.5, patience=3, monitor='val_loss', verbose=1),
]

print('=== FASE 1: Training head layer (base frozen) ===')
history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,          # Cukup 10 epoch untuk fase 1
    callbacks=callbacks
)

# 9. Fase 2 — Fine-tuning
Membuka (unfreeze) 50 layer terakhir MobileNetV2 agar bisa belajar fitur spesifik dari dataset kucing. Layer awal tetap frozen karena sudah mendeteksi fitur umum (edge, tekstur) yang berlaku universal. Learning rate dikurangi 10x agar tidak merusak bobot pre-trained yang sudah bagus.

In [ ]:
base.trainable = True                    # Buka semua layer base
for layer in base.layers[:-50]:          # Freeze kembali semua kecuali 50 layer terakhir
    layer.trainable = False

trainable = sum(1 for l in base.layers if l.trainable)
print(f'Layer trainable: {trainable} dari {len(base.layers)} layer base')

model.compile(
    optimizer=Adam(LR / 10),             # LR 10x lebih kecil dari fase 1 — wajib untuk fine-tuning
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print('=== FASE 2: Fine-tuning 50 layer terakhir ===')
history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,      # Maksimum 50 epoch — EarlyStopping akan berhenti lebih awal
    callbacks=callbacks
)

# 10. Evaluasi
Mengukur performa model pada validation set dan test set. Val accuracy dipakai selama training untuk memilih model terbaik. Test accuracy adalah angka final yang paling jujur — data test tidak pernah dilihat model sama sekali selama training.

In [ ]:
# Evaluasi pada validation set
val_loss, val_acc = model.evaluate(val_ds, verbose=0)
print(f'Val  Loss    : {val_loss:.4f}')
print(f'Val  Accuracy: {val_acc*100:.2f}%')

# Evaluasi pada test set — angka ini yang dilaporkan sebagai performa model
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f'Test Loss    : {test_loss:.4f}')
print(f'Test Accuracy: {test_acc*100:.2f}%')

# 11. Cek Urutan Kelas & Download Model
Memastikan urutan index kelas model sudah benar sebelum dipakai di backend. Urutan ini HARUS sama persis dengan `CLASS_LABELS` di `backend/app/cnn_service.py`. Setelah download, taruh file `cnn_model.h5` di folder `backend/models/`.

In [ ]:
# Tampilkan mapping index → nama kelas
print('Urutan kelas model (sesuaikan CLASS_LABELS di cnn_service.py):')
for i, name in enumerate(CLASS_NAMES):
    print(f'  {i}: {name}')

print(f'\nModel tersimpan di: {MODEL_OUTPUT}')

# Download model .h5 ke komputer lokal
# Setelah download: taruh di backend/models/cnn_model.h5
from google.colab import files
files.download(MODEL_OUTPUT)